# Neutral-set accessibility

This notebook measures how close random parameter points are to the saved NNSE neutral-set cloud, then compares that distance distribution against several simple null geometries. By default it uses the largest available merged NNSE `.npz` for the selected model.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "NNSE":
    ROOT = ROOT.parent
if not (ROOT / "wsbw_pipeline.py").exists():
    candidates = [Path.cwd() / "WhySystemsBiologyWorks", Path.cwd()]
    ROOT = next(path for path in candidates if (path / "wsbw_pipeline.py").exists())
sys.path.insert(0, str(ROOT))

from nnse_accessibility import (
    AccessibilityConfig,
    choose_biggest_neutral_npz,
    run_accessibility,
    save_accessibility,
)

MODEL_KEY = "chen2004"
# Leave as None to use the largest available neutral-set npz for MODEL_KEY.
NPZ_PATH = None
N_RANDOM_STARTS = 10000
MAX_CLOUD_POINTS = 50000
SEED = 42

if NPZ_PATH is None:
    NPZ_PATH = choose_biggest_neutral_npz(ROOT, MODEL_KEY)

print("ROOT:", ROOT)
print("NPZ_PATH:", NPZ_PATH)
print("N_RANDOM_STARTS:", N_RANDOM_STARTS)
print("MAX_CLOUD_POINTS:", MAX_CLOUD_POINTS)

Coordinates are compared in unit-cube coordinates, i.e. each parameter is divided by twice its wild-type value. This matches the sampling range used in the current GP-map and NNSE code.

In [ ]:
config = AccessibilityConfig(
    npz=str(NPZ_PATH),
    n_random=N_RANDOM_STARTS,
    max_cloud=MAX_CLOUD_POINTS,
    seed=SEED,
)
result = run_accessibility(config)
tag = f"{MODEL_KEY}_accessibility_{NPZ_PATH.parent.name}_Nrand{N_RANDOM_STARTS:g}"
out_npz, out_json = save_accessibility(result, tag=tag)
print("Saved:", out_npz)
print("Saved:", out_json)
print("neutral points total:", result["summary"]["neutral_points_total"])
print("neutral points used:", result["summary"]["neutral_points_used"])
print("dimension:", result["summary"]["dimension"])

In [ ]:
summary = result["summary"]["distance_summary"]
order = ["nnse_cloud", "compact_ball", "compact_box", "covariance_ellipsoid", "shuffled_marginals", "synthetic_tube"]
print(f"{'target':24s} {'median':>10s} {'mean':>10s} {'q05':>10s} {'q95':>10s}")
print("-" * 70)
for name in order:
    stats = summary[name]
    print(f"{name:24s} {stats['median']:10.4g} {stats['mean']:10.4g} {stats['q05']:10.4g} {stats['q95']:10.4g}")

In [ ]:
distances = result["distances"]
colors = {
    "nnse_cloud": "#4C78A8",
    "compact_ball": "#F58518",
    "compact_box": "#E45756",
    "covariance_ellipsoid": "#72B7B2",
    "shuffled_marginals": "#54A24B",
    "synthetic_tube": "#B279A2",
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

ax = axes[0]
for name in order:
    vals = distances[name]
    ax.hist(vals, bins=50, density=True, histtype="step", linewidth=1.8, color=colors[name], label=name.replace("_", " "))
ax.set_xlabel("distance to nearest point")
ax.set_ylabel("density")
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
for name in order:
    vals = np.sort(distances[name])
    y = np.linspace(0, 1, len(vals), endpoint=False)
    ax.plot(vals, y, linewidth=1.8, color=colors[name], label=name.replace("_", " "))
ax.set_xlabel("distance to nearest point")
ax.set_ylabel("cumulative fraction")
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
labels = [name.replace("_", " ") for name in order]
data = [distances[name] for name in order]
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
parts = ax.violinplot(data, showmeans=False, showmedians=True, showextrema=False)
for body, name in zip(parts["bodies"], order):
    body.set_facecolor(colors[name])
    body.set_edgecolor("black")
    body.set_alpha(0.45)
parts["cmedians"].set_color("black")
ax.set_xticks(np.arange(1, len(labels) + 1), labels, rotation=25, ha="right")
ax.set_ylabel("distance to nearest point")
plt.show()

Interpretation: if the NNSE cloud is much more accessible than compact ball/box nulls, then random parameter points are closer to the wild-type neutral component than expected for a localized region of comparable scale. If the shuffled-marginal or covariance null is close to the NNSE result, then much of the accessibility may come from coordinate-wise spread or covariance rather than a more structured extended geometry.